# SARIMAX Electricity Demand Forecasting

## Ontario Electricity Peak-Risk Forecasting

This notebook develops and evaluates SARIMAX models for 24-hour-ahead hourly electricity-demand forecasting for the Downtown and Airport-West study regions.

The modeling strategy is informed by the preceding exploratory data analysis, which identified:

- strong short-term temporal dependence;
- pronounced 24-hour seasonality;
- nonlinear temperature-demand relationships;
- regional differences in demand magnitude and variability.

Model development follows a chronological train-validation-test design to prevent temporal data leakage.

### Modeling Objectives

1. Establish a Seasonal Naïve 24-hour forecasting benchmark.
2. Prepare candidate exogenous predictors.
3. Evaluate a parsimonious set of SARIMAX specifications informed by the EDA and autocorrelation analysis.
4. Compare calendar-only and calendar-plus-weather exogenous feature sets.
5. Select model configurations using validation data only.
6. Evaluate the selected models on the untouched 2025 test period.
7. Diagnose forecast errors and residual behaviour.

The final test period is not used for model selection.

In [2]:
# Imports
from pathlib import Path
import warnings
import gc
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("default")

pd.set_option("display.max_columns", None)

In [3]:
# Project paths

DATA_PATH = Path("../data/processed")

DOWNTOWN_FILE = DATA_PATH / "master_hourly_downtown.csv"
AIRPORT_WEST_FILE = DATA_PATH / "master_hourly_airport_west.csv"

In [4]:
# Load analytical datasets

downtown = pd.read_csv(
    DOWNTOWN_FILE,
    parse_dates=["TIMESTAMP", "date"]
)

airport_west = pd.read_csv(
    AIRPORT_WEST_FILE,
    parse_dates=["TIMESTAMP", "date"]
)

datasets = {
    "DOWNTOWN": downtown,
    "AIRPORT_WEST": airport_west
}

for region, df in datasets.items():
    print(
        region,
        "| Shape:", df.shape,
        "| Start:", df["TIMESTAMP"].min(),
        "| End:", df["TIMESTAMP"].max()
    )

DOWNTOWN | Shape: (43824, 16) | Start: 2021-01-01 00:00:00 | End: 2025-12-31 23:00:00
AIRPORT_WEST | Shape: (43824, 16) | Start: 2021-01-01 00:00:00 | End: 2025-12-31 23:00:00


In [5]:
TRAIN_END = pd.Timestamp("2024-06-30 23:00:00")
VALIDATION_END = pd.Timestamp("2024-12-31 23:00:00")

temporal_splits = {}

for region, df in datasets.items():

    df = df.sort_values("TIMESTAMP").copy()

    train = df[
        df["TIMESTAMP"] <= TRAIN_END
    ].copy()

    validation = df[
        (df["TIMESTAMP"] > TRAIN_END) &
        (df["TIMESTAMP"] <= VALIDATION_END)
    ].copy()

    test = df[
        df["TIMESTAMP"] > VALIDATION_END
    ].copy()

    temporal_splits[region] = {
        "train": train,
        "validation": validation,
        "test": test
    }

In [6]:
for region, splits in temporal_splits.items():

    print("=" * 60)
    print(region)

    for split_name, df in splits.items():

        print(
            split_name,
            "| Rows:", len(df),
            "| Start:", df["TIMESTAMP"].min(),
            "| End:", df["TIMESTAMP"].max()
        )

DOWNTOWN
train | Rows: 30648 | Start: 2021-01-01 00:00:00 | End: 2024-06-30 23:00:00
validation | Rows: 4416 | Start: 2024-07-01 00:00:00 | End: 2024-12-31 23:00:00
test | Rows: 8760 | Start: 2025-01-01 00:00:00 | End: 2025-12-31 23:00:00
AIRPORT_WEST
train | Rows: 30648 | Start: 2021-01-01 00:00:00 | End: 2024-06-30 23:00:00
validation | Rows: 4416 | Start: 2024-07-01 00:00:00 | End: 2024-12-31 23:00:00
test | Rows: 8760 | Start: 2025-01-01 00:00:00 | End: 2025-12-31 23:00:00


## 3. Seasonal Naïve 24-Hour Benchmark

A Seasonal Naïve model is used as the reference benchmark for evaluating subsequent SARIMAX models.

For each hourly observation, the forecast corresponds to the electricity demand observed at the same hour of the previous day:

\[
\hat{y}_t = y_{t-24}
\]

The benchmark is initially evaluated only on the validation period (July–December 2024). The 2025 test period remains untouched during model development and selection.

Performance is measured using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Percentage Error (MAPE)

In [11]:
baseline_results = []
baseline_predictions = {}

for region, df in datasets.items():

    temp = (
        df.sort_values("TIMESTAMP")
          .copy()
    )

    # Same hour of the previous day
    temp["SEASONAL_NAIVE_24H"] = (
        temp["TOTAL_CONSUMPTION"].shift(24)
    )

    # Validation period only
    validation = temp[
        (temp["TIMESTAMP"] > TRAIN_END) &
        (temp["TIMESTAMP"] <= VALIDATION_END)
    ].copy()

    y_true = validation["TOTAL_CONSUMPTION"]
    y_pred = validation["SEASONAL_NAIVE_24H"]

    assert y_pred.notna().all()

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = (
        mean_absolute_percentage_error(
            y_true,
            y_pred
        ) * 100
    )

    baseline_results.append({
        "Region": region,
        "Model": "Seasonal Naive 24H",
        "Validation_Rows": len(validation),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_Percent": mape
    })

    baseline_predictions[region] = validation[
        [
            "TIMESTAMP",
            "TOTAL_CONSUMPTION",
            "SEASONAL_NAIVE_24H"
        ]
    ].copy()

baseline_results = pd.DataFrame(baseline_results)

baseline_results.round(2)

,Region,Model,Validation_Rows,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,4416,1565.56,2236.71,6.69
1,AIRPORT_WEST,Seasonal Naive 24H,4416,2343.19,3658.41,6.85


In [12]:
for region, predictions in baseline_predictions.items():

    print("=" * 60)
    print(region)

    display(
        predictions.head(5)
    )

DOWNTOWN


,TIMESTAMP,TOTAL_CONSUMPTION,SEASONAL_NAIVE_24H
30648,2024-07-01 00:00:00,15044.5,21232.4
30649,2024-07-01 01:00:00,13848.9,19612.8
30650,2024-07-01 02:00:00,12954.8,18468.9
30651,2024-07-01 03:00:00,12501.9,17524.4
30652,2024-07-01 04:00:00,12287.7,16928.9


AIRPORT_WEST


,TIMESTAMP,TOTAL_CONSUMPTION,SEASONAL_NAIVE_24H
30648,2024-07-01 00:00:00,21864.8,32626.7
30649,2024-07-01 01:00:00,20008.0,29250.7
30650,2024-07-01 02:00:00,18654.9,26795.9
30651,2024-07-01 03:00:00,17920.1,24973.4
30652,2024-07-01 04:00:00,17602.2,23583.4


In [9]:
baseline_results.round(2)

,Region,Model,Validation_Rows,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,4416,1565.56,2236.71,6.69
1,AIRPORT_WEST,Seasonal Naive 24H,4416,2343.19,3658.41,6.85


In [14]:
for region, df in datasets.items():

    temp = df.sort_values("TIMESTAMP").copy()

first_validation_time = TRAIN_END + pd.Timedelta(hours=1)

previous_day_time = (
        first_validation_time - pd.Timedelta(hours=24)
    )

actual_previous_day = temp.loc[
        temp["TIMESTAMP"] == previous_day_time,
        "TOTAL_CONSUMPTION"
    ].iloc[0]

predicted_value = baseline_predictions[
        region
    ].iloc[0]["SEASONAL_NAIVE_24H"]

print(
        region,
        "| Source timestamp:", previous_day_time,
        "| Previous-day actual:", actual_previous_day,
        "| Prediction:", predicted_value,
        "| Match:",
        np.isclose(actual_previous_day, predicted_value)
    )

AIRPORT_WEST | Source timestamp: 2024-06-30 00:00:00 | Previous-day actual: 32626.7 | Prediction: 32626.7 | Match: True


## Exogenous Feature Design

Candidate exogenous predictors are restricted to variables that have a plausible relationship with electricity demand and are relevant to the 24-hour forecasting objective.

Calendar variables with cyclical structure are transformed using sine and cosine representations rather than treated as continuous integers. This preserves relationships such as hour 23 being adjacent to hour 0 and December being adjacent to January.

Weather variables are evaluated separately because, in an operational 24-hour forecasting setting, future temperature and humidity would need to come from a weather forecast rather than from observed future conditions. This distinction is considered when interpreting the weather-informed SARIMAX results.

In [15]:
def create_sarimax_features(df):

    data = df.copy()

    # Cyclical hour representation
    data["hour_sin"] = np.sin(
        2 * np.pi * data["hour"] / 24
    )

    data["hour_cos"] = np.cos(
        2 * np.pi * data["hour"] / 24
    )

    # Cyclical weekday representation
    data["weekday_sin"] = np.sin(
        2 * np.pi * data["weekday"] / 7
    )

    data["weekday_cos"] = np.cos(
        2 * np.pi * data["weekday"] / 7
    )

    # Cyclical month representation
    data["month_sin"] = np.sin(
        2 * np.pi * (data["month"] - 1) / 12
    )

    data["month_cos"] = np.cos(
        2 * np.pi * (data["month"] - 1) / 12
    )

    return data

In [16]:
modeling_datasets = {
    region: create_sarimax_features(df)
    for region, df in datasets.items()
}

In [17]:
calendar_features = [
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "is_workday",
    "is_public_holiday"
]

In [18]:
for region, df in modeling_datasets.items():

    df["Temp_Squared"] = (
        df["Temp (°C)"] ** 2
    )

In [19]:
weather_features = calendar_features + [
    "Temp (°C)",
    "Temp_Squared",
    "Rel Hum (%)"
]

In [20]:
feature_validation = []

for region, df in modeling_datasets.items():

    for feature_set_name, features in {
        "Calendar": calendar_features,
        "Calendar_Weather": weather_features
    }.items():

        feature_validation.append({
            "Region": region,
            "Feature_Set": feature_set_name,
            "Feature_Count": len(features),
            "Missing_Values":
                df[features].isna().sum().sum(),
            "Infinite_Values":
                np.isinf(
                    df[features].to_numpy(dtype=float)
                ).sum()
        })

feature_validation = pd.DataFrame(
    feature_validation
)

feature_validation

,Region,Feature_Set,Feature_Count,Missing_Values,Infinite_Values
0,DOWNTOWN,Calendar,8,0,0
1,DOWNTOWN,Calendar_Weather,11,0,0
2,AIRPORT_WEST,Calendar,8,0,0
3,AIRPORT_WEST,Calendar_Weather,11,0,0


## SARIMAX Candidate Specifications

Based on the stationarity, ACF, PACF, and seasonal-differencing diagnostics, model development begins with a parsimonious candidate space rather than an exhaustive grid search.

The initial specifications use:

- d = 0, supported by the ADF results;
- D = 1 and s = 24, supported by the strong reduction in daily autocorrelation after 24-hour seasonal differencing;
- p = 1 or 2, reflecting the dominant short-term PACF structure;
- low-order MA and seasonal AR components to limit unnecessary model complexity.

Model selection is based primarily on validation forecasting performance rather than information criteria or ACF/PACF patterns alone.

In [21]:
sarimax_candidates = {
    "C1": {
        "order": (1, 0, 0),
        "seasonal_order": (0, 1, 0, 24)
    },
    "C2": {
        "order": (2, 0, 0),
        "seasonal_order": (0, 1, 0, 24)
    },
    "C3": {
        "order": (2, 0, 1),
        "seasonal_order": (0, 1, 0, 24)
    },
    "C4": {
        "order": (2, 0, 1),
        "seasonal_order": (1, 1, 0, 24)
    }
}

pd.DataFrame(sarimax_candidates).T

,order,seasonal_order
C1,"(1, 0, 0)","(0, 1, 0, 24)"
C2,"(2, 0, 0)","(0, 1, 0, 24)"
C3,"(2, 0, 1)","(0, 1, 0, 24)"
C4,"(2, 0, 1)","(1, 1, 0, 24)"


In [22]:
downtown_model = (
    modeling_datasets["DOWNTOWN"]
    .sort_values("TIMESTAMP")
    .copy()
)

train_dw = downtown_model[
    downtown_model["TIMESTAMP"] <= TRAIN_END
].copy()

validation_dw = downtown_model[
    (downtown_model["TIMESTAMP"] > TRAIN_END) &
    (downtown_model["TIMESTAMP"] <= VALIDATION_END)
].copy()

y_train_dw = train_dw["TOTAL_CONSUMPTION"]

X_train_dw = train_dw[
    calendar_features
].astype(float)

X_validation_dw = validation_dw[
    calendar_features
].astype(float)

print("y_train:", y_train_dw.shape)
print("X_train:", X_train_dw.shape)
print("X_validation:", X_validation_dw.shape)

y_train: (30648,)
X_train: (30648, 8)
X_validation: (4416, 8)


In [23]:
c1 = sarimax_candidates["C1"]

start_time = time.time()

model_c1_dw = SARIMAX(
    endog=y_train_dw,
    exog=X_train_dw,
    order=c1["order"],
    seasonal_order=c1["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c1_dw = model_c1_dw.fit(
    disp=False,
    maxiter=100
)

c1_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c1_dw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c1_dw.mle_retvals.get("iterations")
)

print("AIC:", round(result_c1_dw.aic, 2))
print("BIC:", round(result_c1_dw.bic, 2))
print(f"Execution time: {c1_fit_time / 60:.2f} minutes")

Converged: True
Iterations: 21
AIC: 465474.66
BIC: 465557.95
Execution time: 2.07 minutes


In [24]:
first_24h_exog = X_validation_dw.iloc[:24].copy()

forecast_24h = result_c1_dw.get_forecast(
    steps=24,
    exog=first_24h_exog
)

pred_24h = forecast_24h.predicted_mean

first_forecast = pd.DataFrame({
    "TIMESTAMP": validation_dw["TIMESTAMP"].iloc[:24].values,
    "ACTUAL": validation_dw["TOTAL_CONSUMPTION"].iloc[:24].values,
    "SARIMAX_C1": pred_24h.values
})

first_forecast

,TIMESTAMP,ACTUAL,SARIMAX_C1
0,2024-07-01 00:00:00,15044.5,14724.722324
1,2024-07-01 01:00:00,13848.9,13283.731554
2,2024-07-01 02:00:00,12954.8,12313.620719
3,2024-07-01 03:00:00,12501.9,11538.219895
4,2024-07-01 04:00:00,12287.7,11107.255649
5,2024-07-01 05:00:00,12650.2,11225.851134
6,2024-07-01 06:00:00,13641.6,12239.526177
7,2024-07-01 07:00:00,15397.0,14161.597372
8,2024-07-01 08:00:00,17228.5,15417.778167
9,2024-07-01 09:00:00,19055.0,16174.878949


In [25]:
print("Rows:", len(first_forecast))
print(
    "Start:",
    first_forecast["TIMESTAMP"].min()
)
print(
    "End:",
    first_forecast["TIMESTAMP"].max()
)
print(
    "Missing predictions:",
    first_forecast["SARIMAX_C1"].isna().sum()
)

first_forecast.head()

Rows: 24
Start: 2024-07-01 00:00:00
End: 2024-07-01 23:00:00
Missing predictions: 0


,TIMESTAMP,ACTUAL,SARIMAX_C1
0,2024-07-01 00:00:00,15044.5,14724.722324
1,2024-07-01 01:00:00,13848.9,13283.731554
2,2024-07-01 02:00:00,12954.8,12313.620719
3,2024-07-01 03:00:00,12501.9,11538.219895
4,2024-07-01 04:00:00,12287.7,11107.255649


## 6. Rolling 24-Hour Validation

SARIMAX candidates are evaluated using a rolling 24-hour forecasting procedure.

At each forecast origin, the model generates predictions for the following 24 hourly periods. After the actual observations for that day become available, the model state is updated with those newly observed values while keeping the estimated parameters fixed before generating the next 24-hour forecast.

This procedure simulates repeated day-ahead forecasting while avoiding the computational cost of fully re-estimating model parameters every day.

Model parameters remain fixed during validation. Validation observations are used only to update the model state after they become historically available.

In [26]:
def rolling_24h_validation(
    fitted_result,
    validation_df,
    exog_features,
    horizon=24
):
    
    current_result = fitted_result
    forecasts = []

    validation_df = (
        validation_df
        .sort_values("TIMESTAMP")
        .copy()
    )

    for start in range(0, len(validation_df), horizon):

        block = validation_df.iloc[
            start:start + horizon
        ].copy()

        X_future = (
            block[exog_features]
            .astype(float)
        )

        # Forecast next 24 hours
        forecast = current_result.get_forecast(
            steps=len(block),
            exog=X_future
        )

        block_predictions = pd.DataFrame({
            "TIMESTAMP": block["TIMESTAMP"].values,
            "ACTUAL": block["TOTAL_CONSUMPTION"].values,
            "PREDICTED": forecast.predicted_mean.values
        })

        forecasts.append(block_predictions)

        # Actual observations become available after the forecast
        y_new = block["TOTAL_CONSUMPTION"]
        X_new = block[exog_features].astype(float)

        current_result = current_result.append(
            endog=y_new,
            exog=X_new,
            refit=False
        )

    return pd.concat(
        forecasts,
        ignore_index=True
    )

**Implementation note:**  
Rolling validation uses `append(..., refit=False)` to update the model state while keeping estimated parameters fixed. Because this approach can increase memory usage as the validation history grows, model predictions and metrics are persisted immediately after evaluation and heavy SARIMAX objects are released before the next candidate is trained.

In [28]:
MODEL_OUTPUT_PATH = DATA_PATH / "model_outputs"
MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

start_time = time.time()

c1_dw_rolling = rolling_24h_validation(
    fitted_result=result_c1_dw,
    validation_df=validation_dw,
    exog_features=calendar_features,
    horizon=24
)

c1_rolling_time = time.time() - start_time

print("Rows:", len(c1_dw_rolling))
print("Start:", c1_dw_rolling["TIMESTAMP"].min())
print("End:", c1_dw_rolling["TIMESTAMP"].max())
print(
    "Missing predictions:",
    c1_dw_rolling["PREDICTED"].isna().sum()
)
print(
    f"Execution time: {c1_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c1_dw_rolling.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c1_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 26.29 minutes
Predictions saved successfully.


In [29]:
c1_mae = mean_absolute_error(
    c1_dw_rolling["ACTUAL"],
    c1_dw_rolling["PREDICTED"]
)

c1_rmse = np.sqrt(
    mean_squared_error(
        c1_dw_rolling["ACTUAL"],
        c1_dw_rolling["PREDICTED"]
    )
)

c1_mape = (
    mean_absolute_percentage_error(
        c1_dw_rolling["ACTUAL"],
        c1_dw_rolling["PREDICTED"]
    ) * 100
)

c1_validation_results = pd.DataFrame([{
    "Region": "DOWNTOWN",
    "Model": "SARIMAX C1",
    "Feature_Set": "Calendar",
    "Order": "(1,0,0)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c1_mae,
    "RMSE": c1_rmse,
    "MAPE_Percent": c1_mape,
    "Fit_Time_Minutes": c1_fit_time / 60,
    "Rolling_Time_Minutes": c1_rolling_time / 60
}])

c1_validation_results.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c1_calendar_validation_metrics.csv",
    index=False
)

c1_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,DOWNTOWN,SARIMAX C1,Calendar,"(1,0,0)","(0,1,0,24)",1214.36,1954.46,4.96,2.07,26.29


In [30]:
# Free memory after C1 evaluation

for obj_name in [
    "model_c1_dw",
    "result_c1_dw",
    "forecast_24h",
    "pred_24h",
    "first_forecast"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("C1 heavy objects cleared.")

C1 heavy objects cleared.


In [32]:
del c1_dw_rolling
gc.collect()

0

In [33]:
comparison_c1 = pd.concat([
    baseline_results[
        baseline_results["Region"] == "DOWNTOWN"
    ][
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c1_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ]
], ignore_index=True)

comparison_c1.round(2)

,Region,Model,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,1565.56,2236.71,6.69
1,DOWNTOWN,SARIMAX C1,1214.36,1954.46,4.96


### Initial Validation Result

The first SARIMAX candidate outperformed the Seasonal Naïve 24-hour benchmark across all three evaluation metrics for the Downtown region.

SARIMAX C1, specified as SARIMAX(1,0,0)(0,1,0,24) with calendar exogenous variables, achieved:

- MAE: 1,214.36
- RMSE: 1,954.46
- MAPE: 4.96%

Compared with the Seasonal Naïve benchmark, this represents approximately:

- 22.4% lower MAE
- 12.6% lower RMSE
- 25.9% lower MAPE

These results show that C1 improves validation forecasting performance beyond simply using demand from the same hour of the previous day.

However, C1 is not yet considered the final model. Additional low-complexity candidate specifications will be evaluated using the same validation framework, while the 2025 test period remains untouched.

In [34]:
c2 = sarimax_candidates["C2"]

start_time = time.time()

model_c2_dw = SARIMAX(
    endog=y_train_dw,
    exog=X_train_dw,
    order=c2["order"],
    seasonal_order=c2["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c2_dw = model_c2_dw.fit(
    disp=False,
    maxiter=100
)

c2_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c2_dw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c2_dw.mle_retvals.get("iterations")
)

print("AIC:", round(result_c2_dw.aic, 2))
print("BIC:", round(result_c2_dw.bic, 2))
print(f"Execution time: {c2_fit_time / 60:.2f} minutes")

Converged: True
Iterations: 9
AIC: 454492.77
BIC: 454584.4
Execution time: 1.13 minutes


In [35]:
start_time = time.time()

c2_dw_rolling = rolling_24h_validation(
    fitted_result=result_c2_dw,
    validation_df=validation_dw,
    exog_features=calendar_features,
    horizon=24
)

c2_rolling_time = time.time() - start_time

print("Rows:", len(c2_dw_rolling))
print(
    "Start:",
    c2_dw_rolling["TIMESTAMP"].min()
)
print(
    "End:",
    c2_dw_rolling["TIMESTAMP"].max()
)
print(
    "Missing predictions:",
    c2_dw_rolling["PREDICTED"].isna().sum()
)
print(
    f"Execution time: {c2_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c2_dw_rolling.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c2_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 17.23 minutes
Predictions saved successfully.


In [36]:
c2_mae = mean_absolute_error(
    c2_dw_rolling["ACTUAL"],
    c2_dw_rolling["PREDICTED"]
)

c2_rmse = np.sqrt(
    mean_squared_error(
        c2_dw_rolling["ACTUAL"],
        c2_dw_rolling["PREDICTED"]
    )
)

c2_mape = (
    mean_absolute_percentage_error(
        c2_dw_rolling["ACTUAL"],
        c2_dw_rolling["PREDICTED"]
    ) * 100
)

c2_validation_results = pd.DataFrame([{
    "Region": "DOWNTOWN",
    "Model": "SARIMAX C2",
    "Feature_Set": "Calendar",
    "Order": "(2,0,0)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c2_mae,
    "RMSE": c2_rmse,
    "MAPE_Percent": c2_mape,
    "Fit_Time_Minutes": c2_fit_time / 60,
    "Rolling_Time_Minutes": c2_rolling_time / 60
}])

c2_validation_results.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c2_calendar_validation_metrics.csv",
    index=False
)

c2_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,DOWNTOWN,SARIMAX C2,Calendar,"(2,0,0)","(0,1,0,24)",1231.23,1911.02,5.02,1.13,17.23


In [37]:
comparison_c2 = pd.concat([
    baseline_results[
        baseline_results["Region"] == "DOWNTOWN"
    ][
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c1_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c2_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ]
], ignore_index=True)

comparison_c2.round(2)


,Region,Model,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,1565.56,2236.71,6.69
1,DOWNTOWN,SARIMAX C1,1214.36,1954.46,4.96
2,DOWNTOWN,SARIMAX C2,1231.23,1911.02,5.02


In [38]:
# Free memory after C2 evaluation

for obj_name in [
    "model_c2_dw",
    "result_c2_dw",
    "c2_dw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("C2 heavy objects cleared.")

C2 heavy objects cleared.


In [39]:
c3 = sarimax_candidates["C3"]

start_time = time.time()

model_c3_dw = SARIMAX(
    endog=y_train_dw,
    exog=X_train_dw,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c3_dw = model_c3_dw.fit(
    disp=False,
    maxiter=100
)

c3_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_dw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_dw.mle_retvals.get("iterations")
)

print("AIC:", round(result_c3_dw.aic, 2))
print("BIC:", round(result_c3_dw.bic, 2))

print(
    f"Execution time: {c3_fit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 44
AIC: 454045.89
BIC: 454145.85
Execution time: 2.57 minutes


In [40]:
start_time = time.time()

c3_dw_rolling = rolling_24h_validation(
    fitted_result=result_c3_dw,
    validation_df=validation_dw,
    exog_features=calendar_features,
    horizon=24
)

c3_rolling_time = time.time() - start_time

print("Rows:", len(c3_dw_rolling))
print(
    "Start:",
    c3_dw_rolling["TIMESTAMP"].min()
)
print(
    "End:",
    c3_dw_rolling["TIMESTAMP"].max()
)
print(
    "Missing predictions:",
    c3_dw_rolling["PREDICTED"].isna().sum()
)
print(
    f"Execution time: {c3_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c3_dw_rolling.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c3_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 12.71 minutes
Predictions saved successfully.


In [41]:
c3_mae = mean_absolute_error(
    c3_dw_rolling["ACTUAL"],
    c3_dw_rolling["PREDICTED"]
)

c3_rmse = np.sqrt(
    mean_squared_error(
        c3_dw_rolling["ACTUAL"],
        c3_dw_rolling["PREDICTED"]
    )
)

c3_mape = (
    mean_absolute_percentage_error(
        c3_dw_rolling["ACTUAL"],
        c3_dw_rolling["PREDICTED"]
    ) * 100
)

c3_validation_results = pd.DataFrame([{
    "Region": "DOWNTOWN",
    "Model": "SARIMAX C3",
    "Feature_Set": "Calendar",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c3_mae,
    "RMSE": c3_rmse,
    "MAPE_Percent": c3_mape,
    "Fit_Time_Minutes": c3_fit_time / 60,
    "Rolling_Time_Minutes": c3_rolling_time / 60
}])

c3_validation_results.to_csv(
    MODEL_OUTPUT_PATH / "downtown_c3_calendar_validation_metrics.csv",
    index=False
)

c3_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,DOWNTOWN,SARIMAX C3,Calendar,"(2,0,1)","(0,1,0,24)",1200.14,1891.68,4.85,2.57,12.71


In [42]:
comparison_c3 = pd.concat([
    baseline_results[
        baseline_results["Region"] == "DOWNTOWN"
    ][["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]],

    c1_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c2_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c3_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ]
], ignore_index=True)

comparison_c3.round(2)

,Region,Model,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,1565.56,2236.71,6.69
1,DOWNTOWN,SARIMAX C1,1214.36,1954.46,4.96
2,DOWNTOWN,SARIMAX C2,1231.23,1911.02,5.02
3,DOWNTOWN,SARIMAX C3,1200.14,1891.68,4.85


### C4 Computational Feasibility Assessment

C4 extends C3 by introducing an additional seasonal autoregressive term, resulting in a SARIMAX(2,0,1)(1,1,0,24) specification.

During initial experimentation, C4 successfully converged and produced lower information criteria than the preceding candidates (AIC = 450,528.92; BIC = 450,637.20).

However, full rolling 24-hour validation could not be completed within the available computational environment because the additional seasonal state substantially increased memory requirements during sequential model-state updates.

Because C4 could not be evaluated across the complete 4,416-hour validation period under the same protocol as C1–C3, it is excluded from model selection. Its lower AIC/BIC alone is not considered sufficient evidence of superior out-of-sample forecasting performance.

### C4 — Computational Feasibility Assessment

**NOT EXECUTED IN THE FINAL PIPELINE**

C4 was evaluated during initial experimentation but excluded from the final executable pipeline because full rolling validation exceeded the available computational resources.

```python
# ARCHIVED — DO NOT EXECUTE

c4 = sarimax_candidates["C4"]

model_c4_dw = SARIMAX(
    endog=y_train_dw,
    exog=X_train_dw,
    order=c4["order"],
    seasonal_order=c4["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c4_dw = model_c4_dw.fit(
    disp=False,
    maxiter=100
)
```

Previous experimental result:

- Converged: True
- AIC: 450528.92
- BIC: 450637.20

C4 was not included in final model selection because it could not be evaluated across the complete 4,416-hour validation period using the same rolling-validation protocol as C1–C3.

An alternative state-update implementation using `extend()` was also evaluated to reduce memory usage. However, it required a different index architecture and did not provide a practical solution within the available computational environment. The experiment was therefore discontinued.

In [43]:
# Free memory after C3 evaluation

for obj_name in [
    "model_c3_dw",
    "result_c3_dw",
    "c3_dw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("C3 heavy objects cleared.")

C3 heavy objects cleared.


### C4 Computational Feasibility Assessment

SARIMAX C4, specified as SARIMAX(2,0,1)(1,1,0,24), successfully converged during initial experimentation and produced lower information criteria than the preceding candidates:

- AIC: 450,528.92
- BIC: 450,637.20

However, the additional seasonal autoregressive component substantially increased the computational requirements of the state-space model. Under the available computational environment, attempts to perform the complete rolling 24-hour validation resulted in excessive memory usage and impractical execution times.

Because C4 could not be evaluated across the complete 4,416-hour validation period using the same rolling forecasting protocol applied to C1–C3, it was excluded from model selection. Its lower AIC and BIC values alone are not sufficient evidence of superior out-of-sample forecasting performance.

The best fully validated specification at this stage is therefore SARIMAX C3:

**SARIMAX(2,0,1)(0,1,0,24)**

with calendar exogenous variables.

C3 achieved the best validation performance among the fully evaluated candidates:

- MAE: 1,200.14
- RMSE: 1,891.68
- MAPE: 4.85%

This decision prioritizes comparable out-of-sample validation performance, computational feasibility, and model parsimony rather than selecting a specification based solely on in-sample information criteria.

In [45]:
# Prepare Downtown training and validation data
# for Calendar + Weather SARIMAX

downtown_weather_model = (
    modeling_datasets["DOWNTOWN"]
    .sort_values("TIMESTAMP")
    .copy()
)

train_dw_weather = downtown_weather_model[
    downtown_weather_model["TIMESTAMP"] <= TRAIN_END
].copy()

validation_dw_weather = downtown_weather_model[
    (downtown_weather_model["TIMESTAMP"] > TRAIN_END) &
    (downtown_weather_model["TIMESTAMP"] <= VALIDATION_END)
].copy()

y_train_dw_weather = train_dw_weather[
    "TOTAL_CONSUMPTION"
]

X_train_dw_weather = train_dw_weather[
    weather_features
].astype(float)

X_validation_dw_weather = validation_dw_weather[
    weather_features
].astype(float)

print("y_train:", y_train_dw_weather.shape)
print("X_train:", X_train_dw_weather.shape)
print("X_validation:", X_validation_dw_weather.shape)

y_train: (30648,)
X_train: (30648, 11)
X_validation: (4416, 11)


In [46]:
c3 = sarimax_candidates["C3"]

start_time = time.time()

model_c3_weather_dw = SARIMAX(
    endog=y_train_dw_weather,
    exog=X_train_dw_weather,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c3_weather_dw = model_c3_weather_dw.fit(
    disp=False,
    maxiter=100
)

c3_weather_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_weather_dw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_weather_dw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c3_weather_dw.mle_retvals.get("warnflag")
)

print("AIC:", round(result_c3_weather_dw.aic, 2))
print("BIC:", round(result_c3_weather_dw.bic, 2))

print(
    f"Execution time: {c3_weather_fit_time / 60:.2f} minutes"
)

c:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Converged: False
Iterations: 100
Warnflag: 1
AIC: 452278.85
BIC: 452403.8
Execution time: 7.43 minutes


In [ ]:
print(result_c3_weather_dw.mle_retvals)

In [47]:
start_time = time.time()

result_c3_weather_dw_v2 = model_c3_weather_dw.fit(
    start_params=result_c3_weather_dw.params,
    disp=False,
    maxiter=200
)

c3_weather_refit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_weather_dw_v2.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_weather_dw_v2.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c3_weather_dw_v2.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(result_c3_weather_dw_v2.aic, 2)
)

print(
    "BIC:",
    round(result_c3_weather_dw_v2.bic, 2)
)

print(
    f"Execution time: {c3_weather_refit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 1
Warnflag: 0
AIC: 452278.85
BIC: 452403.8
Execution time: 0.31 minutes


In [48]:
start_time = time.time()

c3_weather_dw_rolling = rolling_24h_validation(
    fitted_result=result_c3_weather_dw_v2,
    validation_df=validation_dw_weather,
    exog_features=weather_features,
    horizon=24
)

c3_weather_rolling_time = time.time() - start_time

print("Rows:", len(c3_weather_dw_rolling))

print(
    "Start:",
    c3_weather_dw_rolling["TIMESTAMP"].min()
)

print(
    "End:",
    c3_weather_dw_rolling["TIMESTAMP"].max()
)

print(
    "Missing predictions:",
    c3_weather_dw_rolling["PREDICTED"].isna().sum()
)

print(
    f"Execution time: {c3_weather_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c3_weather_dw_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "downtown_c3_calendar_weather_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 5.29 minutes
Predictions saved successfully.


In [49]:
# Evaluate C3 Calendar + Weather on validation period

c3_weather_mae = mean_absolute_error(
    c3_weather_dw_rolling["ACTUAL"],
    c3_weather_dw_rolling["PREDICTED"]
)

c3_weather_rmse = np.sqrt(
    mean_squared_error(
        c3_weather_dw_rolling["ACTUAL"],
        c3_weather_dw_rolling["PREDICTED"]
    )
)

c3_weather_mape = (
    mean_absolute_percentage_error(
        c3_weather_dw_rolling["ACTUAL"],
        c3_weather_dw_rolling["PREDICTED"]
    ) * 100
)

c3_weather_validation_results = pd.DataFrame([{
    "Region": "DOWNTOWN",
    "Model": "SARIMAX C3 + Weather",
    "Feature_Set": "Calendar + Weather",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c3_weather_mae,
    "RMSE": c3_weather_rmse,
    "MAPE_Percent": c3_weather_mape,
    "Fit_Time_Minutes": (
        c3_weather_fit_time + c3_weather_refit_time
    ) / 60,
    "Rolling_Time_Minutes": c3_weather_rolling_time / 60
}])

# Persist validation metrics
c3_weather_validation_results.to_csv(
    MODEL_OUTPUT_PATH /
    "downtown_c3_calendar_weather_validation_metrics.csv",
    index=False
)

c3_weather_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,DOWNTOWN,SARIMAX C3 + Weather,Calendar + Weather,"(2,0,1)","(0,1,0,24)",1121.49,1763.92,4.55,7.74,5.29


In [50]:
# Consolidated Downtown validation comparison

downtown_validation_comparison = pd.concat([
    baseline_results[
        baseline_results["Region"] == "DOWNTOWN"
    ][
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c1_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c2_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c3_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c3_weather_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ]
], ignore_index=True)

downtown_validation_comparison.round(2)

,Region,Model,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,1565.56,2236.71,6.69
1,DOWNTOWN,SARIMAX C1,1214.36,1954.46,4.96
2,DOWNTOWN,SARIMAX C2,1231.23,1911.02,5.02
3,DOWNTOWN,SARIMAX C3,1200.14,1891.68,4.85
4,DOWNTOWN,SARIMAX C3 + Weather,1121.49,1763.92,4.55


### Downtown Validation Model Selection

Validation results show a progressive improvement over the Seasonal Naïve 24-hour benchmark.

Among the calendar-only SARIMAX specifications, C3 — SARIMAX(2,0,1)(0,1,0,24) — achieved the best overall validation performance, with an MAE of 1,200.14, RMSE of 1,891.68, and MAPE of 4.85%.

Adding weather information to the C3 specification further improved all three forecasting metrics:

- MAE: 1,121.49
- RMSE: 1,763.92
- MAPE: 4.55%

Compared with C3 using calendar variables only, the Calendar + Weather specification reduced MAE by approximately 6.6%, RMSE by 6.8%, and MAPE by 6.2%.

Therefore, **SARIMAX C3 with Calendar + Weather is selected as the best-performing Downtown specification on the validation period**.

However, this result uses observed weather variables during validation. In an operational 24-hour-ahead forecasting environment, equivalent future weather observations would not be known at forecast time and would need to be replaced by weather forecasts. Therefore, the Calendar + Weather result represents the predictive value of weather information under an observed-weather scenario and should not be interpreted as a fully operational forecasting setup.

The 2025 test period remains untouched and will only be evaluated after model selection has been completed for both study regions.

In [51]:
# Free memory after Downtown C3 + Weather evaluation

for obj_name in [
    "model_c3_weather_dw",
    "result_c3_weather_dw",
    "result_c3_weather_dw_v2",
    "c3_weather_dw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Downtown C3 + Weather heavy objects cleared.")

Downtown C3 + Weather heavy objects cleared.


## 7. Airport-West Model Development and Validation

The Airport-West region is modeled separately because the EDA identified differences in demand magnitude, variability, temperature sensitivity, and temporal behaviour relative to Downtown.

The same chronological training and validation periods and the same 24-hour rolling forecasting protocol are maintained to ensure methodological consistency across regions.

The previously defined SARIMAX candidate structure is used as a starting point, but model performance is evaluated independently for Airport-West rather than assuming that the Downtown-selected specification transfers directly.

The 2025 test period remains untouched.

In [52]:
airport_model = (
    modeling_datasets["AIRPORT_WEST"]
    .sort_values("TIMESTAMP")
    .copy()
)

train_aw = airport_model[
    airport_model["TIMESTAMP"] <= TRAIN_END
].copy()

validation_aw = airport_model[
    (airport_model["TIMESTAMP"] > TRAIN_END) &
    (airport_model["TIMESTAMP"] <= VALIDATION_END)
].copy()

y_train_aw = train_aw["TOTAL_CONSUMPTION"]

X_train_aw = train_aw[
    calendar_features
].astype(float)

X_validation_aw = validation_aw[
    calendar_features
].astype(float)

print("y_train:", y_train_aw.shape)
print("X_train:", X_train_aw.shape)
print("X_validation:", X_validation_aw.shape)

y_train: (30648,)
X_train: (30648, 8)
X_validation: (4416, 8)


In [53]:
# Fit SARIMAX C3 Calendar — Airport-West

c3 = sarimax_candidates["C3"]

start_time = time.time()

model_c3_aw = SARIMAX(
    endog=y_train_aw,
    exog=X_train_aw,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c3_aw = model_c3_aw.fit(
    disp=False,
    maxiter=100
)

c3_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c3_aw.mle_retvals.get("warnflag")
)

print("AIC:", round(result_c3_aw.aic, 2))
print("BIC:", round(result_c3_aw.bic, 2))

print(
    f"Execution time: {c3_aw_fit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 37
Warnflag: 0
AIC: 476609.96
BIC: 476709.91
Execution time: 2.36 minutes


In [54]:
# Rolling 24-hour validation — Airport-West C3 Calendar

start_time = time.time()

c3_aw_rolling = rolling_24h_validation(
    fitted_result=result_c3_aw,
    validation_df=validation_aw,
    exog_features=calendar_features,
    horizon=24
)

c3_aw_rolling_time = time.time() - start_time

print("Rows:", len(c3_aw_rolling))

print(
    "Start:",
    c3_aw_rolling["TIMESTAMP"].min()
)

print(
    "End:",
    c3_aw_rolling["TIMESTAMP"].max()
)

print(
    "Missing predictions:",
    c3_aw_rolling["PREDICTED"].isna().sum()
)

print(
    f"Execution time: {c3_aw_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c3_aw_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c3_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 8.89 minutes
Predictions saved successfully.


In [55]:
# Evaluate C3 Calendar — Airport-West validation

c3_aw_mae = mean_absolute_error(
    c3_aw_rolling["ACTUAL"],
    c3_aw_rolling["PREDICTED"]
)

c3_aw_rmse = np.sqrt(
    mean_squared_error(
        c3_aw_rolling["ACTUAL"],
        c3_aw_rolling["PREDICTED"]
    )
)

c3_aw_mape = (
    mean_absolute_percentage_error(
        c3_aw_rolling["ACTUAL"],
        c3_aw_rolling["PREDICTED"]
    ) * 100
)

c3_aw_validation_results = pd.DataFrame([{
    "Region": "AIRPORT_WEST",
    "Model": "SARIMAX C3",
    "Feature_Set": "Calendar",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c3_aw_mae,
    "RMSE": c3_aw_rmse,
    "MAPE_Percent": c3_aw_mape,
    "Fit_Time_Minutes": c3_aw_fit_time / 60,
    "Rolling_Time_Minutes": c3_aw_rolling_time / 60
}])

c3_aw_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,AIRPORT_WEST,SARIMAX C3,Calendar,"(2,0,1)","(0,1,0,24)",2022.68,3369.53,5.69,2.36,8.89


In [56]:
# Prepare Airport-West Calendar + Weather exogenous variables

y_train_aw_weather = train_aw[
    "TOTAL_CONSUMPTION"
]

X_train_aw_weather = train_aw[
    weather_features
].astype(float)

X_validation_aw_weather = validation_aw[
    weather_features
].astype(float)

print("y_train:", y_train_aw_weather.shape)
print("X_train:", X_train_aw_weather.shape)
print("X_validation:", X_validation_aw_weather.shape)

print(
    "Missing train exog:",
    X_train_aw_weather.isna().sum().sum()
)

print(
    "Missing validation exog:",
    X_validation_aw_weather.isna().sum().sum()
)

y_train: (30648,)
X_train: (30648, 11)
X_validation: (4416, 11)
Missing train exog: 0
Missing validation exog: 0


In [57]:
# Fit SARIMAX C3 Calendar + Weather — Airport-West

c3 = sarimax_candidates["C3"]

start_time = time.time()

model_c3_weather_aw = SARIMAX(
    endog=y_train_aw_weather,
    exog=X_train_aw_weather,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c3_weather_aw = model_c3_weather_aw.fit(
    disp=False,
    maxiter=100
)

c3_weather_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_weather_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_weather_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c3_weather_aw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(result_c3_weather_aw.aic, 2)
)

print(
    "BIC:",
    round(result_c3_weather_aw.bic, 2)
)

print(
    f"Execution time: {c3_weather_aw_fit_time / 60:.2f} minutes"
)

c:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Converged: False
Iterations: 100
Warnflag: 1
AIC: 475982.01
BIC: 476106.95
Execution time: 7.42 minutes


In [58]:
# Continue optimization from the previous Airport-West fit

start_time = time.time()

result_c3_weather_aw_v2 = model_c3_weather_aw.fit(
    start_params=result_c3_weather_aw.params,
    disp=False,
    maxiter=200
)

c3_weather_aw_refit_time = time.time() - start_time

print(
    "Converged:",
    result_c3_weather_aw_v2.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c3_weather_aw_v2.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c3_weather_aw_v2.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(result_c3_weather_aw_v2.aic, 2)
)

print(
    "BIC:",
    round(result_c3_weather_aw_v2.bic, 2)
)

print(
    f"Execution time: {c3_weather_aw_refit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 123
Warnflag: 0
AIC: 474554.53
BIC: 474679.47
Execution time: 7.85 minutes


In [59]:
# Rolling 24-hour validation — Airport-West C3 + Weather

start_time = time.time()

c3_weather_aw_rolling = rolling_24h_validation(
    fitted_result=result_c3_weather_aw_v2,
    validation_df=validation_aw,
    exog_features=weather_features,
    horizon=24
)

c3_weather_aw_rolling_time = time.time() - start_time

print("Rows:", len(c3_weather_aw_rolling))
print(
    "Start:",
    c3_weather_aw_rolling["TIMESTAMP"].min()
)
print(
    "End:",
    c3_weather_aw_rolling["TIMESTAMP"].max()
)
print(
    "Missing predictions:",
    c3_weather_aw_rolling["PREDICTED"].isna().sum()
)
print(
    f"Execution time: {c3_weather_aw_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
c3_weather_aw_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c3_calendar_weather_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 8.46 minutes
Predictions saved successfully.


In [60]:
# Evaluate C3 Calendar + Weather — Airport-West validation

c3_weather_aw_mae = mean_absolute_error(
    c3_weather_aw_rolling["ACTUAL"],
    c3_weather_aw_rolling["PREDICTED"]
)

c3_weather_aw_rmse = np.sqrt(
    mean_squared_error(
        c3_weather_aw_rolling["ACTUAL"],
        c3_weather_aw_rolling["PREDICTED"]
    )
)

c3_weather_aw_mape = (
    mean_absolute_percentage_error(
        c3_weather_aw_rolling["ACTUAL"],
        c3_weather_aw_rolling["PREDICTED"]
    ) * 100
)

c3_weather_aw_validation_results = pd.DataFrame([{
    "Region": "AIRPORT_WEST",
    "Model": "SARIMAX C3 + Weather",
    "Feature_Set": "Calendar + Weather",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c3_weather_aw_mae,
    "RMSE": c3_weather_aw_rmse,
    "MAPE_Percent": c3_weather_aw_mape,

    # Total training cost = initial attempt + continuation
    "Fit_Time_Minutes": (
        c3_weather_aw_fit_time +
        c3_weather_aw_refit_time
    ) / 60,

    "Rolling_Time_Minutes":
        c3_weather_aw_rolling_time / 60
}])

# Persist metrics
c3_weather_aw_validation_results.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c3_calendar_weather_validation_metrics.csv",
    index=False
)

c3_weather_aw_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,AIRPORT_WEST,SARIMAX C3 + Weather,Calendar + Weather,"(2,0,1)","(0,1,0,24)",1941.62,3186.76,5.48,15.27,8.46


In [61]:
# Free memory after Airport-West C3 + Weather evaluation

for obj_name in [
    "model_c3_weather_aw",
    "result_c3_weather_aw",
    "result_c3_weather_aw_v2",
    "c3_weather_aw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Airport-West C3 + Weather heavy objects cleared.")

Airport-West C3 + Weather heavy objects cleared.


In [62]:
for obj_name in [
    "model_c3_aw",
    "result_c3_aw",
    "c3_aw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Airport-West C3 heavy objects cleared.")

Airport-West C3 heavy objects cleared.


In [63]:
# Fit SARIMAX C1 Calendar — Airport-West

c1 = sarimax_candidates["C1"]

start_time = time.time()

model_c1_aw = SARIMAX(
    endog=y_train_aw,
    exog=X_train_aw,
    order=c1["order"],
    seasonal_order=c1["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c1_aw = model_c1_aw.fit(
    disp=False,
    maxiter=100
)

c1_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c1_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c1_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c1_aw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(result_c1_aw.aic, 2)
)

print(
    "BIC:",
    round(result_c1_aw.bic, 2)
)

print(
    f"Execution time: {c1_aw_fit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 27
Warnflag: 0
AIC: 492699.31
BIC: 492782.6
Execution time: 4.33 minutes


In [64]:
# Rolling 24-hour validation — Airport-West C1 Calendar

start_time = time.time()

c1_aw_rolling = rolling_24h_validation(
    fitted_result=result_c1_aw,
    validation_df=validation_aw,
    exog_features=calendar_features,
    horizon=24
)

c1_aw_rolling_time = time.time() - start_time

print("Rows:", len(c1_aw_rolling))

print(
    "Start:",
    c1_aw_rolling["TIMESTAMP"].min()
)

print(
    "End:",
    c1_aw_rolling["TIMESTAMP"].max()
)

print(
    "Missing predictions:",
    c1_aw_rolling["PREDICTED"].isna().sum()
)

print(
    f"Execution time: {c1_aw_rolling_time / 60:.2f} minutes"
)

# Persist predictions
c1_aw_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c1_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 21.14 minutes
Predictions saved successfully.


In [65]:
# Evaluate C1 Calendar — Airport-West validation

c1_aw_mae = mean_absolute_error(
    c1_aw_rolling["ACTUAL"],
    c1_aw_rolling["PREDICTED"]
)

c1_aw_rmse = np.sqrt(
    mean_squared_error(
        c1_aw_rolling["ACTUAL"],
        c1_aw_rolling["PREDICTED"]
    )
)

c1_aw_mape = (
    mean_absolute_percentage_error(
        c1_aw_rolling["ACTUAL"],
        c1_aw_rolling["PREDICTED"]
    ) * 100
)

c1_aw_validation_results = pd.DataFrame([{
    "Region": "AIRPORT_WEST",
    "Model": "SARIMAX C1",
    "Feature_Set": "Calendar",
    "Order": "(1,0,0)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c1_aw_mae,
    "RMSE": c1_aw_rmse,
    "MAPE_Percent": c1_aw_mape,
    "Fit_Time_Minutes": c1_aw_fit_time / 60,
    "Rolling_Time_Minutes": c1_aw_rolling_time / 60
}])

# Persist metrics
c1_aw_validation_results.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c1_calendar_validation_metrics.csv",
    index=False
)

c1_aw_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,AIRPORT_WEST,SARIMAX C1,Calendar,"(1,0,0)","(0,1,0,24)",2014.96,3459.73,5.73,4.33,21.14


In [67]:
# Free memory after Airport-West C1 evaluation

for obj_name in [
    "model_c1_aw",
    "result_c1_aw",
    "c1_aw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Airport-West C1 heavy objects cleared.")

Airport-West C1 heavy objects cleared.


In [68]:
# Fit SARIMAX C2 Calendar — Airport-West

c2 = sarimax_candidates["C2"]

start_time = time.time()

model_c2_aw = SARIMAX(
    endog=y_train_aw,
    exog=X_train_aw,
    order=c2["order"],
    seasonal_order=c2["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

result_c2_aw = model_c2_aw.fit(
    disp=False,
    maxiter=100
)

c2_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    result_c2_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    result_c2_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    result_c2_aw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(result_c2_aw.aic, 2)
)

print(
    "BIC:",
    round(result_c2_aw.bic, 2)
)

print(
    f"Execution time: {c2_aw_fit_time / 60:.2f} minutes"
)

Converged: True
Iterations: 48
Warnflag: 0
AIC: 476750.44
BIC: 476842.06
Execution time: 4.65 minutes


In [69]:
# Rolling 24-hour validation — Airport-West C2 Calendar

start_time = time.time()

c2_aw_rolling = rolling_24h_validation(
    fitted_result=result_c2_aw,
    validation_df=validation_aw,
    exog_features=calendar_features,
    horizon=24
)

c2_aw_rolling_time = time.time() - start_time

print("Rows:", len(c2_aw_rolling))

print(
    "Start:",
    c2_aw_rolling["TIMESTAMP"].min()
)

print(
    "End:",
    c2_aw_rolling["TIMESTAMP"].max()
)

print(
    "Missing predictions:",
    c2_aw_rolling["PREDICTED"].isna().sum()
)

print(
    f"Execution time: {c2_aw_rolling_time / 60:.2f} minutes"
)

# Persist predictions
c2_aw_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c2_calendar_validation_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Rows: 4416
Start: 2024-07-01 00:00:00
End: 2024-12-31 23:00:00
Missing predictions: 0
Execution time: 7.84 minutes
Predictions saved successfully.


In [70]:
# Evaluate C2 Calendar — Airport-West validation

c2_aw_mae = mean_absolute_error(
    c2_aw_rolling["ACTUAL"],
    c2_aw_rolling["PREDICTED"]
)

c2_aw_rmse = np.sqrt(
    mean_squared_error(
        c2_aw_rolling["ACTUAL"],
        c2_aw_rolling["PREDICTED"]
    )
)

c2_aw_mape = (
    mean_absolute_percentage_error(
        c2_aw_rolling["ACTUAL"],
        c2_aw_rolling["PREDICTED"]
    ) * 100
)

c2_aw_validation_results = pd.DataFrame([{
    "Region": "AIRPORT_WEST",
    "Model": "SARIMAX C2",
    "Feature_Set": "Calendar",
    "Order": "(2,0,0)",
    "Seasonal_Order": "(0,1,0,24)",
    "MAE": c2_aw_mae,
    "RMSE": c2_aw_rmse,
    "MAPE_Percent": c2_aw_mape,
    "Fit_Time_Minutes": c2_aw_fit_time / 60,
    "Rolling_Time_Minutes": c2_aw_rolling_time / 60
}])

# Persist metrics
c2_aw_validation_results.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_c2_calendar_validation_metrics.csv",
    index=False
)

c2_aw_validation_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,MAE,RMSE,MAPE_Percent,Fit_Time_Minutes,Rolling_Time_Minutes
0,AIRPORT_WEST,SARIMAX C2,Calendar,"(2,0,0)","(0,1,0,24)",2057.16,3400.78,5.79,4.65,7.84


In [71]:
# Consolidated Airport-West validation comparison

airport_west_validation_comparison = pd.concat([
    baseline_results[
        baseline_results["Region"] == "AIRPORT_WEST"
    ][
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c1_aw_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c2_aw_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c3_aw_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ],

    c3_weather_aw_validation_results[
        ["Region", "Model", "MAE", "RMSE", "MAPE_Percent"]
    ]
], ignore_index=True)

airport_west_validation_comparison.round(2)

,Region,Model,MAE,RMSE,MAPE_Percent
0,AIRPORT_WEST,Seasonal Naive 24H,2343.19,3658.41,6.85
1,AIRPORT_WEST,SARIMAX C1,2014.96,3459.73,5.73
2,AIRPORT_WEST,SARIMAX C2,2057.16,3400.78,5.79
3,AIRPORT_WEST,SARIMAX C3,2022.68,3369.53,5.69
4,AIRPORT_WEST,SARIMAX C3 + Weather,1941.62,3186.76,5.48


### Airport-West Validation Model Selection

All evaluated SARIMAX specifications improved upon the Seasonal Naïve 24-hour benchmark during the Airport-West validation period.

Among the calendar-only specifications, no single candidate dominated all evaluation metrics. C1 achieved the lowest MAE (2,014.96), while C3 achieved the lowest RMSE (3,369.53) and MAPE (5.69%). This confirms that model selection should not rely on model complexity or information criteria alone.

Adding weather variables to the C3 specification produced the strongest overall validation performance:

- MAE: 1,941.62
- RMSE: 3,186.76
- MAPE: 5.48%

Compared with C3 using calendar variables only, the Calendar + Weather specification reduced MAE by approximately 4.0%, RMSE by 5.4%, and MAPE by 3.7%.

Therefore, **SARIMAX C3 with Calendar + Weather is selected as the best-performing Airport-West specification on the validation period**.

As with Downtown, this experiment uses observed weather variables during validation. In a real 24-hour-ahead forecasting environment, future observed weather would not be available at forecast time and would need to be replaced by weather forecasts. The Calendar + Weather results therefore quantify the predictive value of weather information under an observed-weather scenario rather than representing a fully operational forecasting configuration.

The 2025 test period remains untouched and will be evaluated only after model selection has been finalized.

In [72]:
# Free memory after Airport-West C2 evaluation

for obj_name in [
    "model_c2_aw",
    "result_c2_aw",
    "c2_aw_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Airport-West C2 heavy objects cleared.")

Airport-West C2 heavy objects cleared.


## 8. Final Model Training and Test Evaluation

Model selection was completed exclusively using the July–December 2024 validation period.

For both Downtown and Airport-West, SARIMAX(2,0,1)(0,1,0,24) with Calendar + Weather variables achieved the strongest overall validation performance and was selected for final evaluation.

After model selection, each regional model is re-estimated using all observations available through December 31, 2024. This combines the original training and validation periods and allows the final model to use all historical information available before the test period.

The 2025 data are reserved exclusively for final out-of-sample evaluation and are not used for model selection or hyperparameter adjustment.

Because observed weather is used as an exogenous predictor, the resulting test performance represents an observed-weather scenario. Operational deployment would require replacing future observed weather with weather forecasts available at the forecast origin.

In [73]:
# Prepare final training and test datasets

FINAL_TRAIN_END = pd.Timestamp(
    "2024-12-31 23:00:00"
)

TEST_START = pd.Timestamp(
    "2025-01-01 00:00:00"
)

final_datasets = {}

for region, df in modeling_datasets.items():

    temp = (
        df.sort_values("TIMESTAMP")
        .copy()
    )

    final_train = temp[
        temp["TIMESTAMP"] <= FINAL_TRAIN_END
    ].copy()

    test = temp[
        temp["TIMESTAMP"] >= TEST_START
    ].copy()

    final_datasets[region] = {
        "train": final_train,
        "test": test
    }

    print("=" * 60)
    print(region)

    print(
        "Final train:",
        final_train.shape
    )

    print(
        "Test:",
        test.shape
    )

    print(
        "Train start:",
        final_train["TIMESTAMP"].min()
    )

    print(
        "Train end:",
        final_train["TIMESTAMP"].max()
    )

    print(
        "Test start:",
        test["TIMESTAMP"].min()
    )

    print(
        "Test end:",
        test["TIMESTAMP"].max()
    )

    print(
        "Train/Test overlap:",
        final_train["TIMESTAMP"].max()
        >= test["TIMESTAMP"].min()
    )

DOWNTOWN
Final train: (35064, 23)
Test: (8760, 23)
Train start: 2021-01-01 00:00:00
Train end: 2024-12-31 23:00:00
Test start: 2025-01-01 00:00:00
Test end: 2025-12-31 23:00:00
Train/Test overlap: False
AIRPORT_WEST
Final train: (35064, 23)
Test: (8760, 23)
Train start: 2021-01-01 00:00:00
Train end: 2024-12-31 23:00:00
Test start: 2025-01-01 00:00:00
Test end: 2025-12-31 23:00:00
Train/Test overlap: False


In [74]:
# Prepare final Downtown training and test data

final_train_dw = final_datasets["DOWNTOWN"]["train"]
test_dw = final_datasets["DOWNTOWN"]["test"]

y_final_train_dw = (
    final_train_dw["TOTAL_CONSUMPTION"]
    .astype(float)
)

X_final_train_dw = (
    final_train_dw[weather_features]
    .astype(float)
)

X_test_dw = (
    test_dw[weather_features]
    .astype(float)
)

print("y_final_train:", y_final_train_dw.shape)
print("X_final_train:", X_final_train_dw.shape)
print("X_test:", X_test_dw.shape)

print(
    "Missing final train exog:",
    X_final_train_dw.isna().sum().sum()
)

print(
    "Missing test exog:",
    X_test_dw.isna().sum().sum()
)

y_final_train: (35064,)
X_final_train: (35064, 11)
X_test: (8760, 11)
Missing final train exog: 0
Missing test exog: 0


In [75]:
# Final SARIMAX C3 + Weather fit — Downtown
# Training period: 2021-01-01 through 2024-12-31

c3 = sarimax_candidates["C3"]

start_time = time.time()

final_model_dw = SARIMAX(
    endog=y_final_train_dw,
    exog=X_final_train_dw,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

final_result_dw = final_model_dw.fit(
    disp=False,
    maxiter=100
)

final_dw_fit_time = time.time() - start_time

print(
    "Converged:",
    final_result_dw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    final_result_dw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    final_result_dw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(final_result_dw.aic, 2)
)

print(
    "BIC:",
    round(final_result_dw.bic, 2)
)

print(
    f"Execution time: {final_dw_fit_time / 60:.2f} minutes"
)

c:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Converged: False
Iterations: 100
Warnflag: 1
AIC: 518219.79
BIC: 518346.75
Execution time: 19.06 minutes


In [76]:
# Continue final Downtown optimization from previous parameters

start_time = time.time()

final_result_dw_v2 = final_model_dw.fit(
    start_params=final_result_dw.params,
    disp=False,
    maxiter=200
)

final_dw_refit_time = time.time() - start_time

print(
    "Converged:",
    final_result_dw_v2.mle_retvals.get("converged")
)

print(
    "Iterations:",
    final_result_dw_v2.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    final_result_dw_v2.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(final_result_dw_v2.aic, 2)
)

print(
    "BIC:",
    round(final_result_dw_v2.bic, 2)
)

print(
    f"Execution time: {final_dw_refit_time / 60:.2f} minutes"
)

print(
    "Total fit time:",
    round(
        (final_dw_fit_time + final_dw_refit_time) / 60,
        2
    ),
    "minutes"
)

Converged: True
Iterations: 11
Warnflag: 0
AIC: 518218.36
BIC: 518345.33
Execution time: 1.87 minutes
Total fit time: 20.94 minutes


In [77]:
# Final 2025 rolling test evaluation — Downtown

start_time = time.time()

final_dw_test_rolling = rolling_24h_validation(
    fitted_result=final_result_dw_v2,
    validation_df=test_dw,
    exog_features=weather_features,
    horizon=24
)

final_dw_test_rolling_time = time.time() - start_time

print("Rows:", len(final_dw_test_rolling))

print(
    "Start:",
    final_dw_test_rolling["TIMESTAMP"].min()
)

print(
    "End:",
    final_dw_test_rolling["TIMESTAMP"].max()
)

print(
    "Missing predictions:",
    final_dw_test_rolling["PREDICTED"].isna().sum()
)

print(
    f"Execution time: "
    f"{final_dw_test_rolling_time / 60:.2f} minutes"
)

# Persist final test predictions immediately
final_dw_test_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "downtown_final_c3_calendar_weather_2025_test_predictions.csv",
    index=False
)

print("Final Downtown test predictions saved successfully.")

Rows: 8760
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00
Missing predictions: 0
Execution time: 56.57 minutes
Final Downtown test predictions saved successfully.


In [78]:
# Final 2025 test metrics — Downtown

final_dw_test_mae = mean_absolute_error(
    final_dw_test_rolling["ACTUAL"],
    final_dw_test_rolling["PREDICTED"]
)

final_dw_test_rmse = np.sqrt(
    mean_squared_error(
        final_dw_test_rolling["ACTUAL"],
        final_dw_test_rolling["PREDICTED"]
    )
)

final_dw_test_mape = (
    mean_absolute_percentage_error(
        final_dw_test_rolling["ACTUAL"],
        final_dw_test_rolling["PREDICTED"]
    ) * 100
)

final_dw_test_results = pd.DataFrame([{
    "Region": "DOWNTOWN",
    "Model": "SARIMAX C3 + Weather",
    "Feature_Set": "Calendar + Weather",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "Test_Rows": len(final_dw_test_rolling),
    "MAE": final_dw_test_mae,
    "RMSE": final_dw_test_rmse,
    "MAPE_Percent": final_dw_test_mape,
    "Final_Fit_Time_Minutes": (
        final_dw_fit_time + final_dw_refit_time
    ) / 60,
    "Test_Rolling_Time_Minutes":
        final_dw_test_rolling_time / 60
}])

final_dw_test_results.to_csv(
    MODEL_OUTPUT_PATH /
    "downtown_final_c3_calendar_weather_2025_test_metrics.csv",
    index=False
)

final_dw_test_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,Test_Rows,MAE,RMSE,MAPE_Percent,Final_Fit_Time_Minutes,Test_Rolling_Time_Minutes
0,DOWNTOWN,SARIMAX C3 + Weather,Calendar + Weather,"(2,0,1)","(0,1,0,24)",8760,1169.61,1875.7,4.53,20.94,56.57


### Downtown Final Test Performance

The selected Downtown specification, SARIMAX(2,0,1)(0,1,0,24) with Calendar + Weather variables, was re-estimated using all historical observations through December 31, 2024 and evaluated once on the untouched 2025 test period.

The model achieved:

- MAE: 1,169.61
- RMSE: 1,875.70
- MAPE: 4.53%

Compared with validation performance, MAE increased by approximately 4.3% and RMSE by 6.3%, while MAPE remained essentially stable, decreasing slightly from 4.55% to 4.53%.

The relatively small change between validation and test performance indicates that the selected specification maintained stable out-of-sample forecasting accuracy throughout 2025, with no evidence of a substantial deterioration in relative forecast error.

These results should continue to be interpreted as an observed-weather scenario because actual weather observations are used as exogenous predictors during the test period.

In [79]:
# Free memory after final Downtown test evaluation

for obj_name in [
    "final_model_dw",
    "final_result_dw",
    "final_result_dw_v2",
    "final_dw_test_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Final Downtown heavy objects cleared.")

Final Downtown heavy objects cleared.


In [80]:
# Prepare final Airport-West training and test data

final_train_aw = final_datasets["AIRPORT_WEST"]["train"]
test_aw = final_datasets["AIRPORT_WEST"]["test"]

y_final_train_aw = (
    final_train_aw["TOTAL_CONSUMPTION"]
    .astype(float)
)

X_final_train_aw = (
    final_train_aw[weather_features]
    .astype(float)
)

X_test_aw = (
    test_aw[weather_features]
    .astype(float)
)

print("y_final_train:", y_final_train_aw.shape)
print("X_final_train:", X_final_train_aw.shape)
print("X_test:", X_test_aw.shape)

print(
    "Missing final train exog:",
    X_final_train_aw.isna().sum().sum()
)

print(
    "Missing test exog:",
    X_test_aw.isna().sum().sum()
)

y_final_train: (35064,)
X_final_train: (35064, 11)
X_test: (8760, 11)
Missing final train exog: 0
Missing test exog: 0


In [81]:
# Final SARIMAX C3 + Weather fit — Airport-West
# Training period: 2021-01-01 through 2024-12-31

c3 = sarimax_candidates["C3"]

start_time = time.time()

final_model_aw = SARIMAX(
    endog=y_final_train_aw,
    exog=X_final_train_aw,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

final_result_aw = final_model_aw.fit(
    disp=False,
    maxiter=100
)

final_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    final_result_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    final_result_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    final_result_aw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(final_result_aw.aic, 2)
)

print(
    "BIC:",
    round(final_result_aw.bic, 2)
)

print(
    f"Execution time: {final_aw_fit_time / 60:.2f} minutes"
)

c:\Users\ASUS\Documents\Master of Data Analytics\Capstone\ontario-electricity-peak-risk\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


KeyboardInterrupt: 

In [82]:
gc.collect()

3361

In [83]:
# Final SARIMAX C3 + Weather fit — Airport-West
# Skip covariance calculation to avoid expensive OPG computation

c3 = sarimax_candidates["C3"]

start_time = time.time()

final_model_aw = SARIMAX(
    endog=y_final_train_aw,
    exog=X_final_train_aw,
    order=c3["order"],
    seasonal_order=c3["seasonal_order"],
    trend="n",
    enforce_stationarity=True,
    enforce_invertibility=True
)

final_result_aw = final_model_aw.fit(
    disp=False,
    maxiter=100,
    cov_type="none"
)

final_aw_fit_time = time.time() - start_time

print(
    "Converged:",
    final_result_aw.mle_retvals.get("converged")
)

print(
    "Iterations:",
    final_result_aw.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    final_result_aw.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(final_result_aw.aic, 2)
)

print(
    "BIC:",
    round(final_result_aw.bic, 2)
)

print(
    f"Execution time: {final_aw_fit_time / 60:.2f} minutes"
)

Converged: False
Iterations: 100
Warnflag: 1
AIC: 546883.6
BIC: 547010.56
Execution time: 8.03 minutes


In [84]:
# Continue final Airport-West optimization
# from the parameters reached in the first 100 iterations

start_time = time.time()

final_result_aw_v2 = final_model_aw.fit(
    start_params=final_result_aw.params,
    disp=False,
    maxiter=200,
    cov_type="none"
)

final_aw_refit_time = time.time() - start_time

print(
    "Converged:",
    final_result_aw_v2.mle_retvals.get("converged")
)

print(
    "Iterations:",
    final_result_aw_v2.mle_retvals.get("iterations")
)

print(
    "Warnflag:",
    final_result_aw_v2.mle_retvals.get("warnflag")
)

print(
    "AIC:",
    round(final_result_aw_v2.aic, 2)
)

print(
    "BIC:",
    round(final_result_aw_v2.bic, 2)
)

print(
    f"Execution time: "
    f"{final_aw_refit_time / 60:.2f} minutes"
)

print(
    "Total optimization time:",
    round(
        (final_aw_fit_time + final_aw_refit_time) / 60,
        2
    ),
    "minutes"
)

Converged: True
Iterations: 48
Warnflag: 0
AIC: 546143.69
BIC: 546270.66
Execution time: 3.41 minutes
Total optimization time: 11.44 minutes


In [85]:
# Final 2025 rolling test evaluation — Airport-West

start_time = time.time()

final_aw_test_rolling = rolling_24h_validation(
    fitted_result=final_result_aw_v2,
    validation_df=test_aw,
    exog_features=weather_features,
    horizon=24
)

final_aw_test_rolling_time = time.time() - start_time

print("Rows:", len(final_aw_test_rolling))
print(
    "Start:",
    final_aw_test_rolling["TIMESTAMP"].min()
)
print(
    "End:",
    final_aw_test_rolling["TIMESTAMP"].max()
)
print(
    "Missing predictions:",
    final_aw_test_rolling["PREDICTED"].isna().sum()
)
print(
    f"Execution time: "
    f"{final_aw_test_rolling_time / 60:.2f} minutes"
)

# Persist predictions immediately
final_aw_test_rolling.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_final_c3_calendar_weather_2025_test_predictions.csv",
    index=False
)

print("Final Airport-West test predictions saved successfully.")

Rows: 8760
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00
Missing predictions: 0
Execution time: 45.68 minutes
Final Airport-West test predictions saved successfully.


In [86]:
# Final 2025 test metrics — Airport-West

final_aw_test_mae = mean_absolute_error(
    final_aw_test_rolling["ACTUAL"],
    final_aw_test_rolling["PREDICTED"]
)

final_aw_test_rmse = np.sqrt(
    mean_squared_error(
        final_aw_test_rolling["ACTUAL"],
        final_aw_test_rolling["PREDICTED"]
    )
)

final_aw_test_mape = (
    mean_absolute_percentage_error(
        final_aw_test_rolling["ACTUAL"],
        final_aw_test_rolling["PREDICTED"]
    ) * 100
)

final_aw_test_results = pd.DataFrame([{
    "Region": "AIRPORT_WEST",
    "Model": "SARIMAX C3 + Weather",
    "Feature_Set": "Calendar + Weather",
    "Order": "(2,0,1)",
    "Seasonal_Order": "(0,1,0,24)",
    "Test_Rows": len(final_aw_test_rolling),
    "MAE": final_aw_test_mae,
    "RMSE": final_aw_test_rmse,
    "MAPE_Percent": final_aw_test_mape,
    "Final_Fit_Time_Minutes": (
        final_aw_fit_time + final_aw_refit_time
    ) / 60,
    "Test_Rolling_Time_Minutes":
        final_aw_test_rolling_time / 60
}])

# Persist final test metrics
final_aw_test_results.to_csv(
    MODEL_OUTPUT_PATH /
    "airport_west_final_c3_calendar_weather_2025_test_metrics.csv",
    index=False
)

final_aw_test_results.round(2)

,Region,Model,Feature_Set,Order,Seasonal_Order,Test_Rows,MAE,RMSE,MAPE_Percent,Final_Fit_Time_Minutes,Test_Rolling_Time_Minutes
0,AIRPORT_WEST,SARIMAX C3 + Weather,Calendar + Weather,"(2,0,1)","(0,1,0,24)",8760,1783.59,3048.79,5.05,11.44,45.68


### Airport-West Final Test Performance

The selected Airport-West specification, SARIMAX(2,0,1)(0,1,0,24) with Calendar + Weather variables, was re-estimated using all historical observations through December 31, 2024 and evaluated once on the untouched 2025 test period.

The model achieved:

- MAE: 1,783.59
- RMSE: 3,048.79
- MAPE: 5.05%

Compared with validation performance, MAE decreased by approximately 8.1%, RMSE by 4.3%, and MAPE decreased from 5.48% to 5.05%.

The test results therefore show no deterioration in out-of-sample forecasting performance for Airport-West. However, the improvement relative to validation should not be interpreted as evidence that the model learned from the test period, since the 2025 observations were not used for parameter estimation or model selection.

As with Downtown, these results represent an observed-weather scenario. Operational day-ahead forecasting would require weather forecasts available at the forecast origin rather than realized weather observations.

In [87]:
# Free memory after final Airport-West test evaluation

for obj_name in [
    "final_model_aw",
    "final_result_aw",
    "final_result_aw_v2",
    "final_aw_test_rolling"
]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

print("Final Airport-West heavy objects cleared.")

Final Airport-West heavy objects cleared.


In [88]:
# Seasonal Naive 24H — Final 2025 test benchmark

test_baseline_results = []
test_baseline_predictions = {}

for region, df in modeling_datasets.items():

    temp = (
        df.sort_values("TIMESTAMP")
        .copy()
    )

    # Prediction = consumption at the same hour
    # of the previous day
    temp["SEASONAL_NAIVE_24H"] = (
        temp["TOTAL_CONSUMPTION"].shift(24)
    )

    # Restrict evaluation to untouched 2025 test period
    test = temp[
        (temp["TIMESTAMP"] >= TEST_START) &
        (temp["TIMESTAMP"] <= pd.Timestamp(
            "2025-12-31 23:00:00"
        ))
    ].copy()

    y_true = test["TOTAL_CONSUMPTION"]
    y_pred = test["SEASONAL_NAIVE_24H"]

    # Integrity checks
    assert len(test) == 8760
    assert y_true.notna().all()
    assert y_pred.notna().all()

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = (
        mean_absolute_percentage_error(
            y_true,
            y_pred
        ) * 100
    )

    test_baseline_results.append({
        "Region": region,
        "Model": "Seasonal Naive 24H",
        "Test_Rows": len(test),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_Percent": mape
    })

    test_baseline_predictions[region] = pd.DataFrame({
        "TIMESTAMP": test["TIMESTAMP"].values,
        "ACTUAL": y_true.values,
        "PREDICTED": y_pred.values
    })

test_baseline_results = pd.DataFrame(
    test_baseline_results
)

test_baseline_results.round(2)

,Region,Model,Test_Rows,MAE,RMSE,MAPE_Percent
0,DOWNTOWN,Seasonal Naive 24H,8760,1636.81,2461.24,6.46
1,AIRPORT_WEST,Seasonal Naive 24H,8760,2192.63,3691.76,6.26


In [89]:
# Final 2025 test comparison
# Seasonal Naive vs selected SARIMAX models

final_test_comparison = pd.concat([

    test_baseline_results[
        [
            "Region",
            "Model",
            "MAE",
            "RMSE",
            "MAPE_Percent"
        ]
    ],

    final_dw_test_results[
        [
            "Region",
            "Model",
            "MAE",
            "RMSE",
            "MAPE_Percent"
        ]
    ],

    final_aw_test_results[
        [
            "Region",
            "Model",
            "MAE",
            "RMSE",
            "MAPE_Percent"
        ]
    ]

], ignore_index=True)

# Sort for easier regional comparison
final_test_comparison = (
    final_test_comparison
    .sort_values(
        ["Region", "Model"]
    )
    .reset_index(drop=True)
)

final_test_comparison.round(2)

,Region,Model,MAE,RMSE,MAPE_Percent
0,AIRPORT_WEST,SARIMAX C3 + Weather,1783.59,3048.79,5.05
1,AIRPORT_WEST,Seasonal Naive 24H,2192.63,3691.76,6.26
2,DOWNTOWN,SARIMAX C3 + Weather,1169.61,1875.70,4.53
3,DOWNTOWN,Seasonal Naive 24H,1636.81,2461.24,6.46


### Final Forecasting Model Evaluation

The final selected specification for both regions was SARIMAX(2,0,1)(0,1,0,24) with Calendar + Weather variables.

After model selection using the July–December 2024 validation period, the selected specification was re-estimated using all historical observations from January 2021 through December 2024. The resulting models were then evaluated once on the untouched 2025 test period using 24-hour rolling forecasts.

#### Downtown

The final SARIMAX model achieved:

- MAE: 1,169.61
- RMSE: 1,875.70
- MAPE: 4.53%

The Seasonal Naïve 24H benchmark achieved:

- MAE: 1,636.81
- RMSE: 2,461.24
- MAPE: 6.46%

Relative to the benchmark, SARIMAX reduced MAE by approximately 28.5%, RMSE by 23.8%, and MAPE by 29.9%.

#### Airport-West

The final SARIMAX model achieved:

- MAE: 1,783.59
- RMSE: 3,048.79
- MAPE: 5.05%

The Seasonal Naïve 24H benchmark achieved:

- MAE: 2,192.63
- RMSE: 3,691.76
- MAPE: 6.26%

Relative to the benchmark, SARIMAX reduced MAE by approximately 18.7%, RMSE by 17.4%, and MAPE by 19.3%.

Overall, the selected SARIMAX specification consistently outperformed the Seasonal Naïve benchmark across both regions and all three error metrics on the untouched 2025 test period. The results also remained broadly consistent with validation performance, providing evidence that the selected specification generalized to the final out-of-sample period.

These results represent an observed-weather scenario because realized weather variables were used as exogenous inputs. A production day-ahead implementation would require weather forecasts available at the forecast origin.